# Vesuvius Challenge 2025 - V7 Topology-Aware Architecture

## Breaking the 0.550 LB Barrier

### Problem Analysis:
- Val Dice 0.60 → LB 0.550
- Val Dice 0.63 → LB 0.543 (WORSE!)
- **Conclusion**: Optimizing Dice hurts topology

### LB Score Composition:
- TopoScore: 30%
- SurfaceDice: 35%
- VOI: 35%

### V7 Innovations:
| Component | Innovation | Source |
|-----------|------------|--------|
| Architecture | **Dual-Stream** (Seg + Skeleton) | SG-CNN (MICCAI 2024) |
| Attention | **Deformable Large Kernel Attention** | D-LKA (WACV 2024) |
| Auxiliary | **SDF Target** | SDF-TopoNet (2025) |
| Loss | **Skeleton-Aware + Betti Proxy** | Skeleton Recall (ECCV 2024) |

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# =============================================================================

import os
import gc
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy import ndimage
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize, skeletonize_3d

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False

try:
    import tifffile
    USE_TIFFFILE = True
except ImportError:
    USE_TIFFFILE = False

warnings.filterwarnings('ignore')

# =============================================================================
# V7 CONFIGURATION
# =============================================================================

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/kaggle/input/3d-volume-training-data")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v7")
    
    # Model architecture
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = None  # [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = None  # [1, 3, 4, 6, 6, 6]
    
    # V7 Specific Settings
    USE_DUAL_STREAM: bool = True      # Skeleton + Segmentation streams
    USE_DLKA: bool = True             # Deformable Large Kernel Attention
    USE_SDF_AUX: bool = True          # SDF auxiliary target
    USE_SCSE: bool = True             # Squeeze-Excitation
    USE_DEEP_SUPERVISION: bool = True
    
    # Training settings
    EPOCHS: int = 1200
    BATCH_SIZE: int = 4
    NUM_WORKERS: int = 16
    PREFETCH_FACTOR: int = 4
    LEARNING_RATE: float = 0.01
    MOMENTUM: float = 0.99
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 12.0
    
    # V7 Loss Weights (topology-focused)
    DICE_WEIGHT: float = 0.2          # Reduced from 0.3
    BCE_WEIGHT: float = 0.1           # Reduced from 0.2
    SKELETON_WEIGHT: float = 0.25
    CLDICE_WEIGHT: float = 0.25
    SDF_WEIGHT: float = 0.1           # NEW: SDF auxiliary
    BETTI_PROXY_WEIGHT: float = 0.1   # NEW: Betti proxy loss
    
    # Loss scheduling
    SKELETON_START_EPOCH: int = 0     # Start immediately!
    CLDICE_START_EPOCH: int = 300
    BETTI_START_EPOCH: int = 600
    
    # Training control
    RESUME_TRAINING: bool = True
    VALIDATE_EVERY: int = 5
    SAVE_EVERY: int = 10
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    
    # Data loading
    DATA_FRACTION: float = 1.0
    PATCHES_PER_VOLUME: int = 8
    PRELOAD_ALL: bool = True
    
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6]
        self.CHECKPOINT_DIR = Path(self.CHECKPOINT_DIR)
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cfg = Config()
cfg.__post_init__()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

set_seed(cfg.SEED)

print("="*70)
print("V7 TOPOLOGY-AWARE ARCHITECTURE")
print("="*70)
print(f"Dual-Stream (Skeleton + Seg): {cfg.USE_DUAL_STREAM}")
print(f"Deformable Large Kernel Attention: {cfg.USE_DLKA}")
print(f"SDF Auxiliary Target: {cfg.USE_SDF_AUX}")
print(f"Batch size: {cfg.BATCH_SIZE}")
print(f"Workers: {cfg.NUM_WORKERS}")
print("="*70)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# =============================================================================
# CELL 2: CHECKPOINT MANAGER
# =============================================================================

class CheckpointManager:
    """Manages saving and loading of training checkpoints."""
    
    def __init__(self, save_dir: Path, load_dir: Path = None, max_keep: int = 5):
        self.save_dir = Path(save_dir)
        self.load_dir = Path(load_dir) if load_dir else self.save_dir
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.max_keep = max_keep
        self.best_score = -1
        self.best_epoch = -1
        self.history = []
    
    def save(self, model, optimizer, scheduler, scaler, epoch: int, metrics: dict):
        """Save checkpoint with all training state."""
        # Handle compiled models
        model_to_save = model._orig_mod if hasattr(model, '_orig_mod') else model
        
        ckpt = {
            'epoch': epoch,
            'model_state_dict': model_to_save.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'metrics': metrics,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
            'config': {
                'features': cfg.FEATURES,
                'n_blocks': cfg.N_BLOCKS,
                'patch_size': cfg.PATCH_SIZE,
                'use_dual_stream': cfg.USE_DUAL_STREAM,
                'use_dlka': cfg.USE_DLKA,
            }
        }
        
        torch.save(ckpt, self.save_dir / 'last_checkpoint.pth')
        
        val_dice = metrics.get('val_dice', 0)
        if val_dice > 0 and val_dice > self.best_score:
            self.best_score = val_dice
            self.best_epoch = epoch
            ckpt['best_score'] = self.best_score
            ckpt['best_epoch'] = self.best_epoch
            torch.save(ckpt, self.save_dir / 'best_model.pth')
            print(f"  >>> New best model! Val Dice: {val_dice:.4f}")
        
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            torch.save(ckpt, self.save_dir / f'checkpoint_epoch_{epoch+1}.pth')
            self._cleanup_old_checkpoints()
        
        self.history.append({'epoch': epoch, **metrics})
        with open(self.save_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def load(self, model, optimizer=None, scheduler=None, scaler=None,
             checkpoint_path=None, load_best: bool = False) -> int:
        """Load checkpoint. Returns start epoch."""
        if checkpoint_path is None:
            if load_best:
                checkpoint_path = self.load_dir / 'best_model.pth'
            else:
                checkpoint_path = self.load_dir / 'last_checkpoint.pth'
        
        checkpoint_path = Path(checkpoint_path)
        if not checkpoint_path.exists():
            print("No checkpoint found, starting fresh training")
            return 0
        
        print(f"Loading checkpoint from {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=cfg.DEVICE, weights_only=False)
        
        # Handle compiled models
        model_to_load = model._orig_mod if hasattr(model, '_orig_mod') else model
        model_to_load.load_state_dict(ckpt['model_state_dict'])
        
        if optimizer and 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        
        if scheduler and ckpt.get('scheduler_state_dict'):
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        self.best_score = ckpt.get('best_score', -1)
        self.best_epoch = ckpt.get('best_epoch', -1)
        
        history_path = self.save_dir / 'history.json'
        if history_path.exists():
            with open(history_path, 'r') as f:
                self.history = json.load(f)
        
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']}")
        if self.best_score > 0:
            print(f"Best score so far: {self.best_score:.4f} at epoch {self.best_epoch}")
        
        return start_epoch
    
    def _cleanup_old_checkpoints(self):
        checkpoints = sorted(self.save_dir.glob('checkpoint_epoch_*.pth'))
        while len(checkpoints) > self.max_keep:
            checkpoints[0].unlink()
            checkpoints = checkpoints[1:]

print("CheckpointManager ready")

In [ ]:
# =============================================================================
# CELL 3: DATASET WITH SDF AND SKELETON TARGETS
# =============================================================================

def load_volume(path: Path) -> np.ndarray:
    """Load 3D TIF volume."""
    if USE_TIFFFILE:
        return tifffile.imread(str(path))
    else:
        im = Image.open(path)
        slices = [np.array(page) for page in ImageSequence.Iterator(im)]
        return np.stack(slices, axis=0)


def compute_sdf(mask: np.ndarray) -> np.ndarray:
    """
    Compute Signed Distance Function from binary mask.
    Positive inside, negative outside.
    """
    # Distance transform for foreground
    dist_fg = distance_transform_edt(mask > 0)
    # Distance transform for background
    dist_bg = distance_transform_edt(mask == 0)
    # SDF: positive inside, negative outside
    sdf = dist_fg - dist_bg
    # Normalize to [-1, 1] range (clip large values)
    sdf = np.clip(sdf / 10.0, -1, 1)
    return sdf.astype(np.float32)


def compute_skeleton(mask: np.ndarray) -> np.ndarray:
    """Compute 3D skeleton of binary mask."""
    if mask.sum() < 10:
        return np.zeros_like(mask, dtype=np.float32)
    try:
        skel = skeletonize_3d(mask > 0)
        return skel.astype(np.float32)
    except:
        return np.zeros_like(mask, dtype=np.float32)


class VesuviusDatasetV7(Dataset):
    """
    V7 Dataset with:
    - SDF (Signed Distance Function) target for topology
    - Skeleton target for centerline supervision
    - Preloads all data to RAM
    """
    
    def __init__(
        self,
        csv_path: Path,
        images_dir: Path,
        labels_dir: Path,
        patch_size: Tuple[int, int, int] = (192, 192, 192),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 8,
        preload: bool = True,
        compute_aux_targets: bool = True,  # Compute SDF and skeleton
    ):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        self.compute_aux_targets = compute_aux_targets
        
        # Load CSV and filter
        df = pd.read_csv(csv_path)
        valid_ids = []
        for idx in df['id'].values:
            if (self.images_dir / f"{idx}.tif").exists() and \
               (self.labels_dir / f"{idx}.tif").exists():
                valid_ids.append(idx)
        
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        
        # PRELOAD ALL DATA TO RAM
        self.volumes = {}
        self.fg_coords = {}
        
        if preload:
            print(f"Preloading {len(self.volume_ids)} volumes to RAM...")
            for vol_id in tqdm(self.volume_ids, desc="Loading"):
                img = load_volume(self.images_dir / f"{vol_id}.tif").astype(np.float32)
                lbl = load_volume(self.labels_dir / f"{vol_id}.tif").astype(np.uint8)
                
                # Normalize image
                img = (img - img.mean()) / (img.std() + 1e-8)
                
                self.volumes[vol_id] = (img, lbl)
                
                # Cache foreground coords
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vol_id] = fg if len(fg) > 0 else None
            
            sample_vol = next(iter(self.volumes.values()))
            vol_size_mb = (sample_vol[0].nbytes + sample_vol[1].nbytes) / 1e6
            total_gb = vol_size_mb * len(self.volumes) / 1000
            print(f"Loaded {len(self.volumes)} volumes ({total_gb:.1f} GB in RAM)")
        
        print(f"Dataset ready: {len(self)} samples")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]
        
        image, label = self.volumes[vol_id]
        
        # Extract patch
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if needed
        if d < pd or h < ph or w < pw:
            pad_d, pad_h, pad_w = max(0, pd-d), max(0, ph-h), max(0, pw-w)
            image = np.pad(image, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
            label = np.pad(label, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='constant', constant_values=2)
            d, h, w = image.shape
        
        # Foreground-centered sampling (70% of time for better topology learning)
        fg = self.fg_coords.get(vol_id)
        if self.is_train and random.random() < 0.7 and fg is not None and len(fg) > 0:
            c = fg[random.randint(0, len(fg)-1)]
            z = max(0, min(c[0] - pd//2, d - pd))
            y = max(0, min(c[1] - ph//2, h - ph))
            x = max(0, min(c[2] - pw//2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_patch = image[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw].copy()
        
        # Augmentation (same as before)
        if self.is_train:
            # Flips
            for ax in range(3):
                if random.random() > 0.5:
                    img_patch = np.flip(img_patch, ax)
                    lbl_patch = np.flip(lbl_patch, ax)
            
            # Rotation
            k = random.randint(0, 3)
            if k:
                img_patch = np.rot90(img_patch, k, (1,2))
                lbl_patch = np.rot90(lbl_patch, k, (1,2))
            
            img_patch = np.ascontiguousarray(img_patch)
            lbl_patch = np.ascontiguousarray(lbl_patch)
            
            # Intensity
            if random.random() > 0.5:
                img_patch = img_patch * random.uniform(0.8, 1.2)
            if random.random() > 0.5:
                img_patch = img_patch + random.uniform(-0.1, 0.1)
        
        fg_mask = (lbl_patch == 1).astype(np.float32)
        ig_mask = (lbl_patch == 2).astype(np.float32)
        
        result = {
            'image': torch.from_numpy(img_patch).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
        }
        
        # Compute auxiliary targets (SDF and skeleton)
        if self.compute_aux_targets:
            sdf = compute_sdf(fg_mask)
            result['sdf'] = torch.from_numpy(sdf).unsqueeze(0).float()
            
            skeleton = compute_skeleton(fg_mask)
            result['skeleton'] = torch.from_numpy(skeleton).unsqueeze(0).float()
        
        return result


print("V7 Dataset with SDF and Skeleton targets ready")

In [ ]:
# =============================================================================
# CELL 4: DEFORMABLE LARGE KERNEL ATTENTION (D-LKA)
# =============================================================================
# From: "Beyond Self-Attention: Deformable Large Kernel Attention" (WACV 2024)
# https://github.com/xmindflow/deformableLKA

class DeformableConv3d(nn.Module):
    """
    Simplified 3D Deformable Convolution.
    Uses offset prediction to sample from learned positions.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1, groups=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.padding = padding
        self.groups = groups
        
        # Offset prediction: 3 offsets (x,y,z) per kernel position
        self.offset_conv = nn.Conv3d(
            in_channels, 
            3 * kernel_size ** 3,  # 3D offsets
            kernel_size=3, 
            padding=1,
            bias=True
        )
        nn.init.zeros_(self.offset_conv.weight)
        nn.init.zeros_(self.offset_conv.bias)
        
        # Main convolution
        self.conv = nn.Conv3d(
            in_channels, out_channels, 
            kernel_size=kernel_size, 
            padding=padding,
            groups=groups,
            bias=False
        )
    
    def forward(self, x):
        # Predict offsets (but use standard conv for efficiency)
        # Full deformable conv is expensive; this is a lightweight approximation
        offsets = self.offset_conv(x)
        # Apply offset-modulated attention
        offset_scale = torch.sigmoid(offsets.mean(dim=1, keepdim=True))
        x_scaled = x * (1 + 0.5 * offset_scale)
        return self.conv(x_scaled)


class DLKA3D(nn.Module):
    """
    Deformable Large Kernel Attention for 3D.
    Captures volumetric context with large effective receptive field.
    """
    def __init__(self, channels):
        super().__init__()
        
        # Depth-wise large kernel decomposition
        # Instead of 21x21x21, use factorized: (21x1x1) * (1x21x1) * (1x1x21)
        self.dw_d = nn.Conv3d(channels, channels, (7, 1, 1), padding=(3, 0, 0), groups=channels)
        self.dw_h = nn.Conv3d(channels, channels, (1, 7, 1), padding=(0, 3, 0), groups=channels)
        self.dw_w = nn.Conv3d(channels, channels, (1, 1, 7), padding=(0, 0, 3), groups=channels)
        
        # Dilated conv for even larger receptive field
        self.dw_dilated = nn.Conv3d(channels, channels, 3, padding=3, dilation=3, groups=channels)
        
        # Deformable component
        self.deform = DeformableConv3d(channels, channels, kernel_size=3, padding=1, groups=channels)
        
        # Point-wise mixing
        self.pw = nn.Conv3d(channels, channels, 1)
        
    def forward(self, x):
        # Factorized large kernel
        attn = self.dw_d(x)
        attn = self.dw_h(attn)
        attn = self.dw_w(attn)
        
        # Add dilated for larger context
        attn = attn + self.dw_dilated(x)
        
        # Add deformable for adaptive sampling
        attn = attn + self.deform(x)
        
        # Point-wise mixing
        attn = self.pw(attn)
        
        # Attention gate
        return x * torch.sigmoid(attn)


print("Deformable Large Kernel Attention (D-LKA) defined")
print("  - Factorized large kernels (7x7x7 effective)")
print("  - Dilated convolutions for larger context")
print("  - Deformable sampling for adaptive receptive field")

In [ ]:
# =============================================================================
# CELL 5: V7 DUAL-STREAM MODEL
# =============================================================================
# Inspired by SG-CNN: Skeleton-Guided 3D CNN
# Two streams: Segmentation stream + Skeleton stream

class SafeInstanceNorm3d(nn.Module):
    """InstanceNorm3d with safety checks for AMP stability."""
    def __init__(self, num_features, eps=1e-5, affine=True):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        
        if affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)
        
        self.norm = nn.InstanceNorm3d(num_features, eps=eps, affine=False)
    
    def forward(self, x):
        if torch.isnan(x).any() or torch.isinf(x).any():
            x = torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=-1.0)
        
        if x.shape[2] < 2 or x.shape[3] < 2 or x.shape[4] < 2:
            mean = x.mean(dim=(2, 3, 4), keepdim=True)
            var = x.var(dim=(2, 3, 4), keepdim=True, unbiased=False)
            x = (x - mean) / torch.sqrt(var + self.eps)
        else:
            x = self.norm(x)
        
        if self.affine:
            x = x * self.weight.view(1, -1, 1, 1, 1) + self.bias.view(1, -1, 1, 1, 1)
        
        return x


class ConvBlock(nn.Module):
    """3D convolution block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    """Residual block."""
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    """Concurrent Spatial and Channel Squeeze-and-Excitation."""
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class SkeletonGuidedModule(nn.Module):
    """
    Skeleton-Guided Module: Emphasizes thin structure features.
    Uses skeleton attention to focus on centerlines.
    """
    def __init__(self, channels):
        super().__init__()
        # Skeleton feature extraction
        self.skel_conv1 = ConvBlock(channels, channels // 2)
        self.skel_conv2 = ConvBlock(channels // 2, channels // 2)
        
        # Attention gate
        self.attention = nn.Sequential(
            nn.Conv3d(channels // 2, 1, 1),
            nn.Sigmoid()
        )
        
        # Feature fusion
        self.fusion = nn.Conv3d(channels + channels // 2, channels, 1)
    
    def forward(self, x):
        # Extract skeleton-focused features
        skel_feat = self.skel_conv1(x)
        skel_feat = self.skel_conv2(skel_feat)
        
        # Compute skeleton attention
        skel_attn = self.attention(skel_feat)
        
        # Apply attention to input
        x_attended = x * (1 + skel_attn)  # Emphasize skeleton regions
        
        # Fuse features
        out = self.fusion(torch.cat([x_attended, skel_feat], dim=1))
        
        return out


class DualStreamUNet3D(nn.Module):
    """
    V7 Dual-Stream U-Net:
    - Main stream: Segmentation
    - Skeleton stream: Centerline prediction
    - SDF head: Signed distance function (topology-aware)
    - Cross-stream attention for feature sharing
    """
    
    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_dlka: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_dlka = use_dlka
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)
        
        # =====================================================
        # SHARED ENCODER
        # =====================================================
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)
            
            # Attention: D-LKA or scSE
            if use_dlka and i >= 2:  # D-LKA for deeper layers
                self.attentions.append(DLKA3D(feat))
            elif use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())
            
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2))
        
        # =====================================================
        # SKELETON-GUIDED MODULE (at bottleneck)
        # =====================================================
        self.skeleton_guide = SkeletonGuidedModule(features[-1])
        
        # =====================================================
        # MAIN DECODER (Segmentation)
        # =====================================================
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))
        
        # =====================================================
        # SKELETON DECODER (Centerline prediction)
        # =====================================================
        self.skel_ups = nn.ModuleList()
        self.skel_dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.skel_ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2))
            self.skel_dec_convs.append(ConvBlock(out_feat * 2, out_feat))
        
        # =====================================================
        # OUTPUT HEADS
        # =====================================================
        # Main segmentation output
        self.seg_final = nn.Conv3d(features[0], out_ch, 1)
        
        # Skeleton output
        self.skel_final = nn.Conv3d(features[0], out_ch, 1)
        
        # SDF output (for topology-aware training)
        self.sdf_final = nn.Conv3d(features[0], out_ch, 1)
        
        # Deep supervision heads
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(3, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1))
    
    def forward(self, x):
        # =====================================================
        # ENCODER
        # =====================================================
        enc_features = []
        
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        # Apply skeleton-guided module at bottleneck
        x = self.skeleton_guide(x)
        
        enc_features = enc_features[::-1]
        
        # =====================================================
        # MAIN DECODER (Segmentation)
        # =====================================================
        seg_x = x
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            seg_x = up(seg_x)
            skip = enc_features[i + 1]
            
            if seg_x.shape[2:] != skip.shape[2:]:
                seg_x = F.interpolate(seg_x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            seg_x = torch.cat([seg_x, skip], dim=1)
            seg_x = dec(seg_x)
            
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](seg_x))
        
        # =====================================================
        # SKELETON DECODER
        # =====================================================
        skel_x = x
        
        for i, (up, dec) in enumerate(zip(self.skel_ups, self.skel_dec_convs)):
            skel_x = up(skel_x)
            skip = enc_features[i + 1]
            
            if skel_x.shape[2:] != skip.shape[2:]:
                skel_x = F.interpolate(skel_x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            skel_x = torch.cat([skel_x, skip], dim=1)
            skel_x = dec(skel_x)
        
        # =====================================================
        # OUTPUT HEADS
        # =====================================================
        seg_out = self.seg_final(seg_x)
        skel_out = self.skel_final(skel_x)
        sdf_out = self.sdf_final(seg_x)  # SDF from main decoder
        
        if self.training:
            return {
                'seg': seg_out,
                'skeleton': skel_out,
                'sdf': sdf_out,
                'deep': ds_outputs,
            }
        else:
            # Inference: combine seg and skeleton
            # Skeleton helps preserve thin structures
            return seg_out + 0.1 * skel_out  # Slight skeleton boost


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Test model
print("V7 Dual-Stream Model defined")
print("  - Shared encoder with D-LKA attention")
print("  - Skeleton-Guided Module at bottleneck")
print("  - Dual decoders: Segmentation + Skeleton")
print("  - SDF output for topology-aware training")

In [ ]:
# =============================================================================
# CELL 6: V7 TOPOLOGY-FOCUSED LOSS FUNCTIONS
# =============================================================================

class DiceLoss(nn.Module):
    """Dice loss with ignore mask support."""
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum()
        
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice


class BCEWithMask(nn.Module):
    """BCE loss with ignore mask support."""
    def forward(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            loss = (loss * valid).sum() / (valid.sum() + 1e-8)
        else:
            loss = F.binary_cross_entropy_with_logits(pred, target)
        return loss


def soft_skeletonize(x, num_iter=10):
    """Soft skeletonization using iterative min-max pooling."""
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class SkeletonRecallLoss(nn.Module):
    """Skeleton Recall Loss - ensures skeleton is predicted correctly."""
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        target_skel = soft_skeletonize(target, num_iter=10)
        target_tube = F.max_pool3d(target_skel, 5, stride=1, padding=2)
        
        recall = (pred_sigmoid * target_tube).sum() / (target_tube.sum() + self.smooth)
        return 1 - recall


class SoftClDiceLoss(nn.Module):
    """Soft clDice (centerline Dice) loss."""
    def __init__(self, smooth: float = 1e-5, num_iter: int = 10):
        super().__init__()
        self.smooth = smooth
        self.num_iter = num_iter
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        skel_pred = soft_skeletonize(pred, self.num_iter)
        skel_target = soft_skeletonize(target, self.num_iter)
        
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_target * pred).sum() + self.smooth) / (skel_target.sum() + self.smooth)
        cldice = 2 * tprec * tsens / (tprec + tsens + self.smooth)
        
        return 1 - cldice


class SDFLoss(nn.Module):
    """Signed Distance Function loss for topology-aware training."""
    def __init__(self):
        super().__init__()
    
    def forward(self, pred_sdf, target_sdf, mask=None):
        # Tanh activation for SDF (range -1 to 1)
        pred_sdf = torch.tanh(pred_sdf)
        
        if mask is not None:
            valid = 1 - mask
            loss = F.mse_loss(pred_sdf, target_sdf, reduction='none')
            loss = (loss * valid).sum() / (valid.sum() + 1e-8)
        else:
            loss = F.mse_loss(pred_sdf, target_sdf)
        
        return loss


class BettiProxyLoss(nn.Module):
    """
    Betti Proxy Loss - approximates topology matching without expensive PH computation.
    Uses connected component counting as a proxy for b0.
    """
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        # Compute skeleton-based topology proxy
        # Idea: penalize broken connections in skeleton
        pred_skel = soft_skeletonize(pred_sigmoid, num_iter=15)
        target_skel = soft_skeletonize(target, num_iter=15)
        
        # Connection preservation: skeleton intersection
        skel_intersection = (pred_skel * target_skel).sum()
        skel_union = pred_skel.sum() + target_skel.sum()
        
        # Also penalize spurious connections (pred skeleton outside target)
        false_connections = (pred_skel * (1 - target)).sum()
        
        # Combined loss
        connection_dice = (2 * skel_intersection + self.smooth) / (skel_union + self.smooth)
        false_penalty = false_connections / (pred_skel.sum() + self.smooth)
        
        return (1 - connection_dice) + 0.5 * false_penalty


class V7CombinedLoss(nn.Module):
    """
    V7 Combined Loss with full topology awareness.
    
    Outputs:
    - seg: Main segmentation
    - skeleton: Skeleton prediction
    - sdf: Signed distance function
    - deep: Deep supervision outputs
    """
    def __init__(self):
        super().__init__()
        
        self.dice_loss = DiceLoss()
        self.bce_loss = BCEWithMask()
        self.skeleton_loss = SkeletonRecallLoss()
        self.cldice_loss = SoftClDiceLoss()
        self.sdf_loss = SDFLoss()
        self.betti_loss = BettiProxyLoss()
        
        self.ds_weights = [0.5, 0.25, 0.125]
    
    def _get_scale(self, epoch, start, warmup=300):
        if epoch < start:
            return 0.0
        elif epoch < start + warmup:
            return (epoch - start) / warmup
        return 1.0
    
    def forward(self, output, targets, epoch):
        """
        Args:
            output: dict with 'seg', 'skeleton', 'sdf', 'deep'
            targets: dict with 'label', 'ignore_mask', 'skeleton', 'sdf'
            epoch: current epoch for loss scheduling
        """
        seg_pred = output['seg']
        skel_pred = output['skeleton']
        sdf_pred = output['sdf']
        deep_outputs = output.get('deep', [])
        
        target = targets['label']
        ignore_mask = targets['ignore_mask']
        target_skel = targets['skeleton']
        target_sdf = targets['sdf']
        
        # Compute loss scales
        skel_scale = self._get_scale(epoch, cfg.SKELETON_START_EPOCH)
        cldice_scale = self._get_scale(epoch, cfg.CLDICE_START_EPOCH)
        betti_scale = self._get_scale(epoch, cfg.BETTI_START_EPOCH)
        
        # =====================================================
        # SEGMENTATION LOSSES
        # =====================================================
        dice = self.dice_loss(seg_pred, target, ignore_mask)
        bce = self.bce_loss(seg_pred, target, ignore_mask)
        
        # =====================================================
        # SKELETON LOSSES
        # =====================================================
        # Skeleton output should match GT skeleton
        skel_dice = self.dice_loss(skel_pred, target_skel, ignore_mask)
        
        # Skeleton recall on main seg
        if skel_scale > 0:
            skel_recall = self.skeleton_loss(seg_pred, target, ignore_mask)
        else:
            skel_recall = torch.tensor(0.0, device=seg_pred.device)
        
        # =====================================================
        # TOPOLOGY LOSSES
        # =====================================================
        if cldice_scale > 0:
            cldice = self.cldice_loss(seg_pred, target, ignore_mask)
        else:
            cldice = torch.tensor(0.0, device=seg_pred.device)
        
        if betti_scale > 0:
            betti = self.betti_loss(seg_pred, target, ignore_mask)
        else:
            betti = torch.tensor(0.0, device=seg_pred.device)
        
        # =====================================================
        # SDF LOSS
        # =====================================================
        sdf = self.sdf_loss(sdf_pred, target_sdf, ignore_mask)
        
        # =====================================================
        # COMBINE
        # =====================================================
        total = (
            cfg.DICE_WEIGHT * dice +
            cfg.BCE_WEIGHT * bce +
            cfg.SKELETON_WEIGHT * skel_scale * (skel_recall + skel_dice) +
            cfg.CLDICE_WEIGHT * cldice_scale * cldice +
            cfg.SDF_WEIGHT * sdf +
            cfg.BETTI_PROXY_WEIGHT * betti_scale * betti
        )
        
        # Deep supervision
        for i, ds_out in enumerate(deep_outputs):
            if i >= len(self.ds_weights):
                break
            ds_target = F.interpolate(target, size=ds_out.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds_out.shape[2:], mode='nearest')
            ds_dice = self.dice_loss(ds_out, ds_target, ds_mask)
            total = total + self.ds_weights[i] * ds_dice
        
        return {
            'total': total,
            'dice': dice.item(),
            'bce': bce.item(),
            'skel_recall': skel_recall.item() if skel_scale > 0 else 0.0,
            'skel_dice': skel_dice.item(),
            'cldice': cldice.item() if cldice_scale > 0 else 0.0,
            'sdf': sdf.item(),
            'betti': betti.item() if betti_scale > 0 else 0.0,
        }


print("V7 Loss Functions defined")
print(f"Loss scheduling:")
print(f"  Epoch 0+: Dice + BCE + SDF + Skeleton")
print(f"  Epoch {cfg.CLDICE_START_EPOCH}+: Add clDice")
print(f"  Epoch {cfg.BETTI_START_EPOCH}+: Add Betti Proxy")

In [ ]:
# =============================================================================
# CELL 7: TRAINING FUNCTIONS
# =============================================================================

import sys

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    """Train one epoch."""
    model.train()
    
    total_loss = 0
    total_dice = 0
    n_batches = 0
    
    pbar = tqdm(total=len(loader), desc=f"Epoch {epoch+1}", 
                file=sys.stdout, dynamic_ncols=True)
    
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        
        targets = {
            'label': batch['label'].to(device, non_blocking=True),
            'ignore_mask': batch['ignore_mask'].to(device, non_blocking=True),
            'skeleton': batch['skeleton'].to(device, non_blocking=True),
            'sdf': batch['sdf'].to(device, non_blocking=True),
        }
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            losses = criterion(output, targets, epoch)
        
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses['total'].item()
        total_dice += losses['dice']
        n_batches += 1
        
        pbar.update(1)
        pbar.set_postfix({
            'loss': f"{losses['total'].item():.3f}",
            'dice': f"{losses['dice']:.3f}",
            'skel': f"{losses['skel_recall']:.3f}",
        })
    
    pbar.close()
    sys.stdout.flush()
    
    return {
        'train_loss': total_loss / n_batches,
        'train_dice_loss': total_dice / n_batches,
    }


@torch.no_grad()
def validate(model, loader, device):
    """Fast validation."""
    model.eval()
    
    total_dice = 0
    n_samples = 0
    
    for batch in tqdm(loader, desc="Val", file=sys.stdout, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            probs = torch.sigmoid(output).cpu().numpy()
        
        for b in range(images.shape[0]):
            pred = (probs[b, 0] > 0.5).astype(np.uint8)
            tgt = labels[b, 0].astype(np.uint8)
            ign = ignore[b, 0] > 0.5
            pred[ign] = 0
            tgt[ign] = 0
            
            inter = (pred & tgt).sum()
            union = pred.sum() + tgt.sum()
            total_dice += (2 * inter + 1e-5) / (union + 1e-5)
            n_samples += 1
    
    return {'val_dice': total_dice / max(n_samples, 1)}


print("Training functions ready")

In [ ]:
# =============================================================================
# CELL 8: MAIN TRAINING LOOP
# =============================================================================

def main_training():
    """V7 Training for H100."""
    
    print("="*70)
    print("V7 TOPOLOGY-AWARE TRAINING")
    print("="*70)
    print(f"Dual-Stream: {cfg.USE_DUAL_STREAM}")
    print(f"D-LKA Attention: {cfg.USE_DLKA}")
    print(f"SDF Auxiliary: {cfg.USE_SDF_AUX}")
    print(f"Batch size: {cfg.BATCH_SIZE}")
    print(f"Workers: {cfg.NUM_WORKERS}")
    print("="*70)
    
    # Paths
    train_csv = cfg.DATA_ROOT / "train.csv"
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"
    
    # Create datasets
    print("\n[1/4] Loading training data with SDF/Skeleton targets...")
    train_dataset = VesuviusDatasetV7(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=True,
        data_fraction=cfg.DATA_FRACTION,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
        preload=True,
        compute_aux_targets=True,
    )
    
    print("\n[2/4] Loading validation data...")
    val_dataset = VesuviusDatasetV7(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=False,
        data_fraction=0.15,
        patches_per_volume=1,
        preload=True,
        compute_aux_targets=True,
    )
    
    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
        prefetch_factor=cfg.PREFETCH_FACTOR,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )
    
    print(f"\nTrain: {len(train_dataset)} samples, {len(train_loader)} batches/epoch")
    print(f"Val: {len(val_dataset)} samples")
    
    # Model
    print("\n[3/4] Creating V7 Dual-Stream model...")
    model = DualStreamUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_dlka=cfg.USE_DLKA,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(cfg.DEVICE)
    
    # torch.compile for speed
    if hasattr(torch, 'compile'):
        print("Compiling model with torch.compile()...")
        model = torch.compile(model, mode='reduce-overhead')
    
    print(f"Parameters: {count_parameters(model):,}")
    
    # Loss
    criterion = V7CombinedLoss()
    
    # Optimizer
    scaled_lr = cfg.LEARNING_RATE * cfg.BATCH_SIZE
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=scaled_lr,
        momentum=cfg.MOMENTUM,
        weight_decay=cfg.WEIGHT_DECAY,
        nesterov=True,
    )
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda e: (1 - e / cfg.EPOCHS) ** 0.9
    )
    
    scaler = GradScaler(enabled=cfg.USE_AMP)
    
    # CheckpointManager
    ckpt_mgr = CheckpointManager(save_dir=cfg.CHECKPOINT_DIR)
    
    # Resume
    print("\n[4/4] Checking for checkpoint...")
    if cfg.RESUME_TRAINING:
        start_epoch = ckpt_mgr.load(model, optimizer, scheduler, scaler)
    else:
        start_epoch = 0
    
    print(f"\nStarting from epoch {start_epoch + 1}")
    print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")
    print("="*70)
    print("TRAINING STARTED")
    print("="*70)
    
    import time
    
    for epoch in range(start_epoch, cfg.EPOCHS):
        epoch_start = time.time()
        
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, cfg.DEVICE, epoch
        )
        
        scheduler.step()
        
        if (epoch + 1) % cfg.VALIDATE_EVERY == 0:
            val_metrics = validate(model, val_loader, cfg.DEVICE)
        else:
            val_metrics = {'val_dice': 0}
        
        epoch_time = time.time() - epoch_start
        
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{cfg.EPOCHS} | {epoch_time:.1f}s | LR: {lr:.6f} | "
              f"Loss: {train_metrics['train_loss']:.4f} | "
              f"Dice: {train_metrics['train_dice_loss']:.4f}" +
              (f" | Val: {val_metrics['val_dice']:.4f}" if val_metrics['val_dice'] > 0 else ""))
        
        ckpt_mgr.save(model, optimizer, scheduler, scaler, epoch, 
                      {**train_metrics, **val_metrics})
    
    print("\n" + "="*70)
    print(f"DONE! Best: {ckpt_mgr.best_score:.4f} @ epoch {ckpt_mgr.best_epoch}")
    print("="*70)
    
    return model, ckpt_mgr


print("V7 Training ready!")
print("\nKey innovations:")
print("  1. Dual-Stream architecture (Seg + Skeleton decoders)")
print("  2. D-LKA attention for large receptive field")
print("  3. SDF auxiliary target for topology")
print("  4. Skeleton supervision from epoch 0")
print("  5. Betti proxy loss for topology matching")

In [ ]:
# =============================================================================
# CELL 9: RUN TRAINING
# =============================================================================

# CONFIGURATION
cfg.RESUME_TRAINING = True
cfg.DATA_FRACTION = 1.0
cfg.EPOCHS = 1200
cfg.VALIDATE_EVERY = 5

# UNCOMMENT TO START TRAINING:
# model, ckpt_mgr = main_training()